# TweetyScraper

[`Source`](https://github.com/iragca/capstone-project-2/blob/master/src/scraper/Tweety.py)

An asynchronous scraper for retrieving tweets and user information using the [`tweety`](https://github.com/mahrtayyab/tweety) library. Provides methods for logging in, fetching tweets, BLM-related trends, and normalizing data into the local {ref}`tweet` and {ref}`user` models.

---

## Initialization

In [2]:
from src.scraper import TweetyScraper

scraper = TweetyScraper(previous_session=True)
scraper

````{card}
:class-header: bg-light
:class-card: border-0 shadow-none

Parameters
^^^
`previous_session` (*bool*, default=*True*)
:   Whether to reuse a saved session (`session.json`). If *False*, logs in with credentials from settings.

````

````{card}
:class-header: bg-light
:class-card: border-0 shadow-none

Attributes
^^^
`previous_session` (*bool*, default=*True*)
:   Whether to reuse a saved session (`session.json`). If *False*, logs in with credentials from settings.

````

---

## Public Methods

### `login()`

Authenticate with Twitter and return a [`TwitterAsync`](https://mahrtayyab.github.io/tweety_docs/basic/twitter-class.html#TwitterAsync) client.

Returns
:   [`TwitterAsync`](https://mahrtayyab.github.io/tweety_docs/basic/twitter-class.html#TwitterAsync) — An authenticated Tweety client instance.

Raises
:   `AssertionError` — If username, password, or TOTP are missing in settings when `previous_session=False`.

---

### `get_blm_trends()`

Fetch tweets containing the hashtag `#blacklivesmatter`. Saves results into an Excel file.

Returns
:   *None* — Writes tweets to an XLSX file.

---

### `get_tweets_of_user()`

Fetch tweets of a given user. Supports pagination and wait times between requests.

Parameters
:   `username` (*str*) — The username of the target account.
:   `pages` (*int*, default=100) — Number of result pages to fetch.
:   `wait_time` (*int*, default=30) — Delay between page requests.

Returns
:   *list*\[*dict*] — A list of validated tweets serialized into dicts.
:   *\[]* — Empty list if no tweets are found or errors occur.

Raises
:   `KeyboardInterrupt` — If scraping is manually stopped.
:   `Exception` — For unexpected tweet parsing errors.

---

### `get_user_info()`

Retrieve user information either by `user_id` or `username`.

Parameters
:   `user_id` (*int* | *str* | *None*) — The numeric ID of the user. If a string, will be coerced into an integer.
:   `username` (*str* | *None*) — The username of the user.

Returns
:   {ref}`user` — A validated User model object.
:   *None* — If the user does not exist or errors occur.

Raises
:   `ValueError` — If neither `user_id` nor `username` is provided.
:   `UserNotFound` — If the given user cannot be found by Tweety.

---

## Private Helpers

### `process_tweety_tweet()`

Normalize a `TweetyTweet` object into a {ref}`tweet` dict.

* Adds derived fields such as:

  * `has_blm_hashtag`
  * `status_link`
  * `conversation_id`
  * `retweet_status_id`
  * `quoted_status_id`

Parameters
:   `tweet` (`TweetyTweet`) — A raw tweet object from Tweety.

Returns
:   *dict* — A serialized tweet dictionary matching the {ref}`tweet` model.

Raises
:   `ValueError` — If a `conversation_id` cannot be resolved from the tweet.

---

### `process_tweety_user()`

Normalize a `TweetyUser` object into a {ref}`user`.

Parameters
:   `user` (`TweetyUser`) — A raw user object from Tweety.

Returns
:   {ref}`user` — A validated User model object.

---

## Example Usage

In [ ]:
from src.scraper import TweetyScraper


scraper = TweetyScraper(previous_session=True)

# Get tweets from a user
tweets = await scraper.get_tweets_of_user("elonmusk", pages=1)
print(f"Fetched {len(tweets)} tweets")

# Get user info
user = await scraper.get_user_info(username="elonmusk")
print(user)
